# POMDP Candidate Evaluation

This notebook evaluates the belief-limited attacker over the mixed `70/30` VAE candidate pool. It compares three observation-quality presets: `good_contact`, `baseline_night`, and `poor_contact`.

The selector only sees noisy attacker-facing observations and candidate parameters. It does not use true hit counts, expected loss, or full-state ranks while selecting candidates.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from convoy_sim.realism import get_attacker_observation_config
from experiments.evaluate_attack_candidate_pool import evaluate_candidate_pool
from experiments.run_pomdp_candidate_selector import run_belief_selector


## Configuration

`good_contact` is the default realism preset. The full-state run path points at the latest mixed-VAE final baseline run, so the notebook can compute top-k overlap without rerunning the expensive oracle selector.


In [ ]:
CANDIDATE_PATH = PROJECT_ROOT / "data" / "attack_profiles" / "vae_candidates" / "mixed_curated70_random30_hit_candidates.jsonl"
FULL_STATE_RUN_DIR = PROJECT_ROOT / "results" / "runs" / "candidate_pool_eval" / "20260512_144030_vae_final_baseline_mixed_vae"

OBSERVATION_PRESETS = ["good_contact", "baseline_night", "poor_contact"]
MAX_PROFILES = 1000
TOP_K = 25
SELECTION_SEED = 1945

RUN_SELECTION = True
RUN_MONTE_CARLO_EVAL = True

CONVOY_PROFILE = "convoy_layout_1"
EVAL_SEEDS = [1942, 1943, 1944]
N_TRIALS_PER_SEED = 10
T_MAX = 400.0
OBJECTIVE_CFG = {"preset": "balanced_default"}

OUTPUT_ROOT = Path("results/runs")


In [ ]:
preset_rows = []
for preset in OBSERVATION_PRESETS:
    cfg = get_attacker_observation_config(preset)
    row = {"preset": preset}
    row.update(cfg.to_dict())
    preset_rows.append(row)

pd.DataFrame(preset_rows)


## Belief-Limited Selection

This cell ranks the mixed VAE candidate pool under each observation preset and writes a top-k JSONL candidate pool for each preset. Those JSONL files are the bridge into the existing Monte Carlo evaluator.


In [ ]:
selection_dirs: dict[str, Path] = {}

if RUN_SELECTION:
    for preset in OBSERVATION_PRESETS:
        run_dir = run_belief_selector(
            candidate_path=CANDIDATE_PATH,
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            run_name=f"pomdp_{preset}_notebook",
            convoy_profile=CONVOY_PROFILE,
            max_profiles=MAX_PROFILES,
            top_k=TOP_K,
            seed=SELECTION_SEED,
            observation_preset=preset,
        )
        selection_dirs[preset] = run_dir
        print(f"{preset}: {run_dir}")
else:
    selection_dirs = {
        # "good_contact": PROJECT_ROOT / "results/runs/pomdp_candidate_selector/<existing_good_contact_run>",
        # "baseline_night": PROJECT_ROOT / "results/runs/pomdp_candidate_selector/<existing_baseline_night_run>",
        # "poor_contact": PROJECT_ROOT / "results/runs/pomdp_candidate_selector/<existing_poor_contact_run>",
    }

selection_dirs


In [ ]:
belief_tables = []
for preset, run_dir in selection_dirs.items():
    df = pd.read_csv(run_dir / "belief_ranked_candidates.csv")
    df.insert(0, "preset", preset)
    belief_tables.append(df)

belief_df = pd.concat(belief_tables, ignore_index=True) if belief_tables else pd.DataFrame()
belief_df.head(10)


## Evaluate Selected Top-K Candidates

Each preset's selected top-k candidate pool is evaluated with the same scored Monte Carlo pipeline used by the full-state attacker baseline.


In [ ]:
eval_dirs: dict[str, Path] = {}

if RUN_MONTE_CARLO_EVAL:
    for preset, selection_dir in selection_dirs.items():
        selected_pool = selection_dir / "top_belief_candidate_pool.jsonl"
        run_dir = evaluate_candidate_pool(
            candidate_path=selected_pool,
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            run_name=f"pomdp_{preset}_top{TOP_K}_eval_notebook",
            convoy_profile=CONVOY_PROFILE,
            max_profiles=None,
            top_k=TOP_K,
            seeds=EVAL_SEEDS,
            n_trials_per_seed=N_TRIALS_PER_SEED,
            t_max=T_MAX,
            max_hits_per_torpedo=1,
            objective_cfg=OBJECTIVE_CFG,
        )
        eval_dirs[preset] = run_dir
        print(f"{preset}: {run_dir}")
else:
    eval_dirs = {
        # "good_contact": PROJECT_ROOT / "results/runs/candidate_pool_eval/<existing_good_contact_eval>",
        # "baseline_night": PROJECT_ROOT / "results/runs/candidate_pool_eval/<existing_baseline_night_eval>",
        # "poor_contact": PROJECT_ROOT / "results/runs/candidate_pool_eval/<existing_poor_contact_eval>",
    }

eval_dirs


## Summary Table

For POMDP rows, `candidate_pool` is the belief-selected top-k pool. For the optional full-state row, `top_k` is the oracle-selected top-k from the existing full-state mixed-VAE baseline.


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

summary_rows = []

if FULL_STATE_RUN_DIR.exists():
    metrics = load_json(FULL_STATE_RUN_DIR / "metrics_summary.json")
    top = metrics["top_k"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": "full_state_oracle",
        "preset": "oracle",
        "profiles": int(top.get("profiles", TOP_K)),
        "expected_hits": float(top["expected_hits"]),
        "expected_loss": float(top.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(top.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(top.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(top.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

for preset, run_dir in eval_dirs.items():
    metrics = load_json(run_dir / "metrics_summary.json")
    pool = metrics["candidate_pool"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": "belief_limited",
        "preset": preset,
        "profiles": int(pool.get("profiles", TOP_K)),
        "expected_hits": float(pool["expected_hits"]),
        "expected_loss": float(pool.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(pool.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(pool.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(pool.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
if len(summary_df):
    plot_df = summary_df.copy()
    plot_df["label"] = plot_df["selector"] + "\n" + plot_df["preset"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="white")
    axes[0].bar(plot_df["label"], plot_df["expected_hits"], color="#4c78a8")
    axes[0].set_title("Expected Hits")
    axes[0].set_ylabel("hits")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].bar(plot_df["label"], plot_df["expected_loss"], color="#f58518")
    axes[1].set_title("Expected Loss")
    axes[1].set_ylabel("loss")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


## Rank Overlap With Full-State Oracle

This checks whether noisy belief selection is choosing the same candidates as the full-state Monte Carlo oracle. Lower overlap is expected as observation quality degrades.


In [ ]:
overlap_rows = []
if FULL_STATE_RUN_DIR.exists() and selection_dirs:
    full_ranked = pd.read_csv(FULL_STATE_RUN_DIR / "ranked_candidates.csv")
    oracle_top = set(full_ranked.head(TOP_K)["profile_id"].astype(str))
    for preset, run_dir in selection_dirs.items():
        belief_ranked = pd.read_csv(run_dir / "belief_ranked_candidates.csv")
        belief_top = set(belief_ranked.head(TOP_K)["profile_id"].astype(str))
        overlap_rows.append({
            "preset": preset,
            "top_k": TOP_K,
            "overlap_count": len(oracle_top & belief_top),
            "overlap_rate": len(oracle_top & belief_top) / float(TOP_K),
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df


In [ ]:
if len(overlap_df):
    fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
    ax.bar(overlap_df["preset"], overlap_df["overlap_rate"], color="#54a24b")
    ax.set_ylim(0, 1)
    ax.set_title("Top-K Overlap With Full-State Oracle")
    ax.set_ylabel("overlap rate")
    ax.set_xlabel("observation preset")
    plt.tight_layout()
    plt.show()
